In [1]:
# Current path:
!pwd # Current repo - "/Users/pini/wix_home_assignment"

/Users/pini/wix_home_assignment


# Nexa Treasury Case - python notebook - task B

### Data sources

In [2]:
# In this part, I made sure to use simple but effective python data analysis
# approaches in order to keep things simple, clean and easy to debug/examine (for the sake of this home assignment)

# Relevant libs for this project:
import pandas as pd
import numpy as np

import os

/Users/pini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/pini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [3]:
# Datasets
df_sub = pd.read_csv('source_files/subscription_collections_by_currency.csv')
df_fx = pd.read_csv('source_files/fx_rates.csv')

In [4]:
# Prepare column formats
df_sub.columns = [x.replace(' ', '_') for x in df_sub.columns]
df_fx.columns = [x.lower() for x in df_fx.columns]

### Task B1 - Nexa's forecast net exposure by currency for April to September 2026

In [5]:
# Forecast period Apr-Sep 2026, without USD (the reporting currency has no FX exposure)
df_forecast = df_sub[(df_sub['data_type'] == 'Forecast') & (df_sub['currency'] != 'USD')].copy()

# A missing existing hedge means no hedge in that month
df_forecast['existing_fwd_hedges_sold_lc_000s'] = df_forecast['existing_fwd_hedges_sold_lc_000s'].fillna(0.0)

# Net exposure before hedging: collections - refunds - local costs (positive = Nexa is long the currency, negative = short)
df_forecast['net_exposure_gross_lc_000s'] = (
    df_forecast['gross_collections_lc_000s']
    - df_forecast['refunds_chargebacks_lc_000s']
    - df_forecast['local_cost_payments_lc_000s']
)

# Open exposure after deducting existing hedges (forwards already sold)
df_forecast['net_exposure_lc_000s'] = (
    df_forecast['net_exposure_gross_lc_000s'] - df_forecast['existing_fwd_hedges_sold_lc_000s']
)

print("Forecast Net Exposure in Local Currency (in Thousands)")
# Presentation
display(df_forecast[['month', 'currency', 'gross_collections_lc_000s', 'net_exposure_gross_lc_000s',
                     'existing_fwd_hedges_sold_lc_000s', 'net_exposure_lc_000s']].head(10))

Forecast Net Exposure in Local Currency (in Thousands)


,month,currency,gross_collections_lc_000s,net_exposure_gross_lc_000s,existing_fwd_hedges_sold_lc_000s,net_exposure_lc_000s
30,2026-04,EUR,19459.400,16363.100,6000.0,10363.100
31,2026-05,EUR,19813.200,16693.700,4500.0,12193.700
32,2026-06,EUR,20171.500,17028.700,3000.0,14028.700
33,2026-07,EUR,19527.600,16391.400,0.0,16391.400
34,2026-08,EUR,19480.900,16333.400,0.0,16333.400
35,2026-09,EUR,20454.900,17265.400,0.0,17265.400
48,2026-04,GBP,7783.800,6344.300,2500.0,3844.300
49,2026-05,GBP,7925.300,6475.500,2000.0,4475.500
50,2026-06,GBP,8.069,-1452.131,1500.0,-2952.131
51,2026-07,GBP,7811.000,6352.500,0.0,6352.500


In [6]:
import pandas as pd
import numpy as np

# Latest spot rates (last date in the file, 10 April 2026) - same snapshot date as the other tasks
latest_rate_date = df_fx['rate_date'].max()
df_fx_base = df_fx[df_fx['rate_date'] == latest_rate_date].copy()
print(f"FX snapshot date: {latest_rate_date}")

df_forecast_merged = pd.merge(
    df_forecast,
    df_fx_base[['currency', 'spot_mid', 'quote_convention']],
    on='currency',
    how='left'
)

# Convert to USD by quote convention: "USD per 1" multiply, "per 1 USD" divide
def lc_to_usd(amount_lc, rate, conv):
    if 'USD per 1' in str(conv):
        return amount_lc * rate
    return amount_lc / rate

# Convert the three columns (before hedging, existing hedges, after hedging) to USD thousands
for lc_col, usd_col in [('net_exposure_gross_lc_000s', 'net_exposure_gross_usd_000s'),
                        ('existing_fwd_hedges_sold_lc_000s', 'existing_fwd_hedges_sold_usd_000s'),
                        ('net_exposure_lc_000s', 'net_exposure_usd_000s')]:
    df_forecast_merged[usd_col] = df_forecast_merged.apply(
        lambda r: lc_to_usd(r[lc_col], r['spot_mid'], r['quote_convention']), axis=1)

# Summary per currency across the 6 months
net_exposure_summary = df_forecast_merged.groupby(['currency', 'quote_convention']).agg(
    total_gross_exposure_lc_000s=('net_exposure_gross_lc_000s', 'sum'),
    total_gross_exposure_usd_000s=('net_exposure_gross_usd_000s', 'sum'),
    existing_hedges_lc_000s=('existing_fwd_hedges_sold_lc_000s', 'sum'),
    existing_hedges_usd_000s=('existing_fwd_hedges_sold_usd_000s', 'sum'),
    total_net_exposure_lc_000s=('net_exposure_lc_000s', 'sum'),
    total_net_exposure_usd_000s=('net_exposure_usd_000s', 'sum'),
    avg_monthly_exposure_usd_000s=('net_exposure_gross_usd_000s', 'mean')
).reset_index()

# Policy: a currency with average monthly exposure below USD 1.0M is not hedged
net_exposure_summary['is_material'] = net_exposure_summary['avg_monthly_exposure_usd_000s'].abs() >= 1000

print("Forecast Net FX Exposure Summary (April - Sep 2026) in Thousands")
pd.options.display.float_format = '{:,.2f}'.format
# Presentation
display(net_exposure_summary.sort_values(by='total_gross_exposure_usd_000s', ascending=False))

FX snapshot date: 2026-04-10
Forecast Net FX Exposure Summary (April - Sep 2026) in Thousands


,currency,quote_convention,total_gross_exposure_lc_000s,total_gross_exposure_usd_000s,existing_hedges_lc_000s,existing_hedges_usd_000s,total_net_exposure_lc_000s,total_net_exposure_usd_000s,avg_monthly_exposure_usd_000s,is_material
4,EUR,USD per 1 EUR,"100,075.70","108,602.15","13,500.00","14,650.20","86,575.70","93,951.95","18,100.36",True
5,GBP,USD per 1 GBP,"30,748.47","40,268.20","6,000.00","7,857.60","24,748.47","32,410.60","6,711.37",True
8,JPY,JPY per 1 USD,"4,932,781.00","32,841.42","550,000.00","3,661.78","4,382,781.00","29,179.63","5,473.57",True
1,BRL,BRL per 1 USD,"190,830.30","31,214.06",0.00,0.00,"190,830.30","31,214.06","5,202.34",True
2,CAD,CAD per 1 USD,"40,144.70","29,431.60","2,000.00","1,466.28","38,144.70","27,965.32","4,905.27",True
0,AUD,USD per 1 AUD,"32,951.60","21,715.10",0.00,0.00,"32,951.60","21,715.10","3,619.18",True
9,MXN,MXN per 1 USD,"260,472.50","14,544.95",0.00,0.00,"260,472.50","14,544.95","2,424.16",True
3,CHF,CHF per 1 USD,"4,703.20","5,303.56",0.00,0.00,"4,703.20","5,303.56",883.93,False
7,INR,INR per 1 USD,"372,578.00","4,428.60",0.00,0.00,"372,578.00","4,428.60",738.10,False
11,SEK,SEK per 1 USD,"43,655.60","4,216.75",0.00,0.00,"43,655.60","4,216.75",702.79,False


### Task B2 - Recommend a hedging action for each material currency

In [7]:
# Spot rates and forward points as of 10 April 2026
df_fx_rates_analysis = df_fx_base[[
    'currency', 'currency_pair', 'quote_convention', 'spot_mid',
    'fwd_points_1m', 'fwd_points_3m', 'fwd_points_6m', 'pip_size'
]].copy()

# Forward rate = spot + (forward points * pip size)
df_fx_rates_analysis['fwd_rate_3m'] = df_fx_rates_analysis['spot_mid'] + (
    df_fx_rates_analysis['fwd_points_3m'] * df_fx_rates_analysis['pip_size']
)
df_fx_rates_analysis['fwd_rate_6m'] = df_fx_rates_analysis['spot_mid'] + (
    df_fx_rates_analysis['fwd_points_6m'] * df_fx_rates_analysis['pip_size']
)

# Benefit/cost of selling the currency forward vs spot, in % from a USD point of view:
# for 'USD per 1' currencies a higher forward rate is a benefit; for 'per 1 USD' currencies it is a cost
def seller_benefit_pct(row, fwd_col):
    if 'USD per 1' in row['quote_convention']:
        return (row[fwd_col] / row['spot_mid'] - 1) * 100
    return (row['spot_mid'] / row[fwd_col] - 1) * 100

df_fx_rates_analysis['fwd_cost_benefit_3m_pct'] = df_fx_rates_analysis.apply(seller_benefit_pct, axis=1, fwd_col='fwd_rate_3m')
df_fx_rates_analysis['fwd_cost_benefit_6m_pct'] = df_fx_rates_analysis.apply(seller_benefit_pct, axis=1, fwd_col='fwd_rate_6m')

print("FX Rates & Forward Points Analysis (positive = forward better than spot for a seller of the currency)")
pd.options.display.float_format = '{:,.4f}'.format
# Presentation
display(df_fx_rates_analysis[['currency', 'spot_mid', 'fwd_rate_3m', 'fwd_cost_benefit_3m_pct', 'fwd_cost_benefit_6m_pct']])

FX Rates & Forward Points Analysis (positive = forward better than spot for a seller of the currency)


,currency,spot_mid,fwd_rate_3m,fwd_cost_benefit_3m_pct,fwd_cost_benefit_6m_pct
348,EUR,1.0852,1.0907,0.5050,0.9989
349,GBP,1.3096,1.3104,0.0634,0.1245
350,ILS,3.7547,3.7547,0.0003,0.0000
351,BRL,6.1136,6.2550,-2.2611,-4.3779
352,JPY,150.2000,148.7920,0.9463,1.8906
353,AUD,0.6590,0.6596,0.0865,0.1700
354,CAD,1.3640,1.3589,0.3768,0.7482
355,MXN,17.9081,18.0984,-1.0517,-2.0600
356,INR,84.1300,84.5510,-0.4979,-0.9828
357,PLN,3.9652,3.9751,-0.2498,-0.4931


In [8]:
# Settings from the hedging policy (Appendix B)
RATIO_M1_3 = (0.50, 0.80)
RATIO_M4_6 = (0.25, 0.50)
FWD_COST_LIMIT_PCT = -1.0      # Forward costs more than 1% - stay at the low end of the range
NDF_CURRENCIES = ['BRL', 'INR']  # Capital-control currencies - an NDF needs Director of Treasury approval

reco_rows = []
for _, s in net_exposure_summary.iterrows():
    ccy = s['currency']
    fx_row = df_fx_rates_analysis[df_fx_rates_analysis['currency'] == ccy].iloc[0]
    months = df_forecast_merged[df_forecast_merged['currency'] == ccy].sort_values('month')

    # Negative exposure (ILS) means Nexa buys the currency, so the forward benefit is the opposite of a seller's
    is_short = s['total_gross_exposure_lc_000s'] < 0
    sign = -1 if is_short else 1
    benefit_3m = sign * fx_row['fwd_cost_benefit_3m_pct']
    benefit_6m = sign * fx_row['fwd_cost_benefit_6m_pct']

    if not s['is_material']:
        reco_rows.append({
            'currency': ccy, 'direction': 'Buy' if is_short else 'Sell',
            'avg_monthly_exposure_usd_000s': s['avg_monthly_exposure_usd_000s'], 'is_material': False,
            'existing_cover_pct_m1_3': 0.0, 'target_pct_m1_3': 0.0, 'target_pct_m4_6': 0.0,
            'additional_hedge_lc_000s': 0.0, 'additional_hedge_usd_000s': 0.0,
            'fwd_benefit_3m_pct': benefit_3m, 'fwd_benefit_6m_pct': benefit_6m, 'est_fwd_benefit_usd_000s': 0.0,
            'instrument': 'No hedge - convert at spot as needed',
            'rationale': 'Below USD 1.0M average monthly exposure (policy B.2)'
        })
        continue

    # Pick the hedge ratio: mid-point of the range, or the low end when the forward is expensive
    expensive = min(benefit_3m, benefit_6m) < FWD_COST_LIMIT_PCT
    r13 = RATIO_M1_3[0] if expensive else sum(RATIO_M1_3) / 2
    r46 = RATIO_M4_6[0] if expensive else sum(RATIO_M4_6) / 2

    add_lc, add_usd, benefit_usd = 0.0, 0.0, 0.0
    for i, m in enumerate(months.itertuples()):
        ratio = r13 if i < 3 else r46
        # The target includes existing hedges, so only the gap is added
        extra = max(ratio * abs(m.net_exposure_gross_lc_000s) - m.existing_fwd_hedges_sold_lc_000s, 0)
        extra_usd = lc_to_usd(extra, m.spot_mid, m.quote_convention)
        add_lc += extra
        add_usd += extra_usd
        benefit_usd += extra_usd * (benefit_3m if i < 3 else benefit_6m) / 100

    m13 = months.head(3)
    existing_cover = m13['existing_fwd_hedges_sold_lc_000s'].sum() / abs(m13['net_exposure_gross_lc_000s'].sum()) * 100

    if ccy in NDF_CURRENCIES:
        instrument = 'NDF (capital controls) - needs Director of Treasury approval'
    else:
        instrument = ('Buy' if is_short else 'Sell') + f' {ccy} deliverable forwards (3m for months 1-3, 6m for months 4-6)'

    reco_rows.append({
        'currency': ccy, 'direction': 'Buy' if is_short else 'Sell',
        'avg_monthly_exposure_usd_000s': s['avg_monthly_exposure_usd_000s'], 'is_material': True,
        'existing_cover_pct_m1_3': existing_cover, 'target_pct_m1_3': r13 * 100, 'target_pct_m4_6': r46 * 100,
        'additional_hedge_lc_000s': add_lc, 'additional_hedge_usd_000s': add_usd,
        'fwd_benefit_3m_pct': benefit_3m, 'fwd_benefit_6m_pct': benefit_6m, 'est_fwd_benefit_usd_000s': benefit_usd,
        'instrument': instrument,
        'rationale': ('Forward carry is costly - low end of policy range' if expensive
                      else 'Mid-point of policy range (50-80% / 25-50%)')
    })

df_hedge_reco = pd.DataFrame(reco_rows).sort_values('avg_monthly_exposure_usd_000s', key=abs, ascending=False)

print("Hedging Recommendation (USD in thousands)")
pd.options.display.float_format = '{:,.2f}'.format
# Presentation
display(df_hedge_reco[['currency', 'direction', 'avg_monthly_exposure_usd_000s', 'is_material', 'existing_cover_pct_m1_3',
                       'target_pct_m1_3', 'target_pct_m4_6', 'additional_hedge_usd_000s',
                       'fwd_benefit_3m_pct', 'est_fwd_benefit_usd_000s', 'instrument']])

Hedging Recommendation (USD in thousands)


,currency,direction,avg_monthly_exposure_usd_000s,is_material,existing_cover_pct_m1_3,target_pct_m1_3,target_pct_m4_6,additional_hedge_usd_000s,fwd_benefit_3m_pct,est_fwd_benefit_usd_000s,instrument
4,EUR,Sell,"18,100.36",True,26.95,65.00,37.50,"41,022.62",0.50,307.63,Sell EUR deliverable forwards (3m for months 1...
6,ILS,Buy,"-15,176.19",True,0.00,65.00,37.50,"46,563.28",-0.00,-0.08,Buy ILS deliverable forwards (3m for months 1-...
5,GBP,Sell,"6,711.37",True,52.78,65.00,37.50,"14,537.44",0.06,15.03,Sell GBP deliverable forwards (3m for months 1...
8,JPY,Sell,"5,473.57",True,22.28,65.00,37.50,"13,173.62",0.95,182.76,Sell JPY deliverable forwards (3m for months 1...
1,BRL,Sell,"5,202.34",True,0.00,50.00,25.00,"11,661.79",-2.26,-347.19,NDF (capital controls) - needs Director of Tre...
2,CAD,Sell,"4,905.27",True,9.96,65.00,37.50,"13,618.58",0.38,71.80,Sell CAD deliverable forwards (3m for months 1...
0,AUD,Sell,"3,619.18",True,0.00,65.00,37.50,"11,132.14",0.09,13.02,Sell AUD deliverable forwards (3m for months 1...
9,MXN,Sell,"2,424.16",True,0.00,50.00,25.00,"5,443.72",-1.05,-75.69,Sell MXN deliverable forwards (3m for months 1...
3,CHF,Sell,883.93,False,0.00,0.00,0.00,0.00,1.01,0.00,No hedge - convert at spot as needed
7,INR,Sell,738.10,False,0.00,0.00,0.00,0.00,-0.50,0.00,No hedge - convert at spot as needed


**How exposure and hedging are calculated**

* **Scope:** forecast months only (April to September 2026). USD is left out because it is our reporting currency. A month with no existing hedge counts as zero.
* **Net exposure:** collections minus refunds and chargebacks minus local costs, before hedges. Then I take off the forwards we already sold to get the open exposure. A negative number (ILS) means Nexa is a net buyer of that currency.
* **USD values:** the latest spot rate (10 April 2026). "USD per 1" currencies are multiplied, "per 1 USD" currencies are divided.
* **Materiality:** a currency with average exposure under USD 1.0M a month is not hedged and is converted at spot (policy B.2).
* **Hedge ratios:** 50-80% for months 1-3 and 25-50% for months 4-6, counting hedges already in place (policy B.3). I use the middle of each range, and drop to the low end when the forward is expensive (worse than -1%).
* **Forward cost or benefit:** forward rate = spot + forward points x pip size. The percentage is shown for the hedging side in USD terms, so positive means the forward beats spot. For ILS, where Nexa is the buyer, the sign is flipped.
* **NDFs:** BRL and INR have capital controls, so any NDF needs written approval from the Director of Treasury (policy B.4).

### Task B3 - What would change the recommendation
* A forecast move of more than about 10% in collections or local costs for a material currency, which means resizing the hedge.
* Forward points that flip a benefit into a cost, or the other way round, which means rechecking horizon and ratio.
* A sharp spot move against Nexa, or a change in hedging policy.
* BRL: whether the Director of Treasury approves an NDF.
* ILS: a change in the payroll plan, or a stronger shekel that makes waiting more expensive.

In [9]:
import pandas as pd
import numpy as np
import os

output_dir = 'analytics_output'
os.makedirs(output_dir, exist_ok=True)

net_exposure_summary.to_csv(f'{output_dir}/forecast_net_fx_exposure_summary.csv', index=False)

df_fx_rates_analysis.to_csv(f'{output_dir}/fx_rates_forward_analysis.csv', index=False)

# Hedging recommendation per currency + monthly detail for the dashboard charts
df_hedge_reco.to_csv(f'{output_dir}/fx_hedging_recommendation.csv', index=False)

df_forecast_merged[['month', 'currency', 'net_exposure_gross_lc_000s', 'existing_fwd_hedges_sold_lc_000s',
                    'net_exposure_lc_000s', 'net_exposure_gross_usd_000s', 'existing_fwd_hedges_sold_usd_000s',
                    'net_exposure_usd_000s']].to_csv(f'{output_dir}/fx_monthly_exposure.csv', index=False)

print(f"Successfully saved to: {output_dir}")

Successfully saved to: analytics_output
